# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [3]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\guowi\AppData\Local\Temp\ipykernel_36020\4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [4]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [5]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [6]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [7]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [8]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata (like `source`, `page`, and `start_index`) is important because it:

- **Enables citation and verification** — lets the application trace each answer back to where it came from.
- **Supports filtered retrieval** — narrows search to relevant subsets of documents.
- **Uniquely identifies chunks** — distinguishes otherwise-similar pieces of text.
- **Powers post-retrieval logic** — such as deduplication, access control, and evaluation.



## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [9]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [10]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

- **Chunk size — precision vs. context completeness:** small chunks give focused, sharply-matching embeddings but may cut off the full answer, while large chunks preserve context but blur the embedding across multiple topics and lower precision.
- **Chunk overlap — boundary safety vs. redundancy and cost:** more overlap keeps ideas that fall on a chunk boundary intact, but duplicates text, increases storage, and can return near-duplicate results.
- **The goal:** pick the smallest chunks that still hold a complete thought, with just enough overlap to avoid splitting key sentences.

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [11]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [12]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [13]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

**What it helps understand:**

- **Ranking** — orders candidate chunks by how close they are to the query in embedding space.

**What it does *not* prove:**

- **Correctness or truth** — a close match isn't necessarily accurate or actually an answer to the question.
- **An absolute measure** — only meaningful relative to other scores from the same model; there's no universal threshold.
- **Genuine relevance** — a high score can come from surface similarity rather than real meaning.



## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [14]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [16]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [17]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [18]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs that may suggest your cat needs veterinary attention include:

- **Pain or anxiety/fear/stress signs**: cowering, crouching, crawling, muscular tension, freezing, hiding, frantic fleeing, flattened or rotated-back ears, dilated pupils, straight-forward whiskers, or defensive vocalizations like hissing, yowling, growling, or screaming [Source 4].
- **Changes in grooming**: increased grooming may point to a skin problem; reduced grooming can signal underlying illness, bladder pain, joint pain, or reduced mobility [Source 1][Source 2].
- **Behavior changes**: increased nocturnal activity, vocalization, or changes in normal habits/activity may indicate a medical issue [Source 3].
- **GI signs**: vomiting, vomiting hairballs, diarrhea, changes in appetite, or weight changes should be discussed with a veterinarian [Source 3].
- **Mobility/pain concerns**: signs of osteoarthritis or other musculoskeletal pain are important to have assessed [Source 1].

If you notice these signs, it’s a 

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [19]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
The guidelines recommend individualized preventive care based on a cat’s age, lifestyle, and risk factors [Source 1][Source 4]. In general, they advise at least annual veterinary examinations for all cats, with more frequent visits as needed; senior cats should be seen at least every 6 months [Source 4]. Preventive care can include routine parasite control, since broad-spectrum products are likely beneficial for most pet cats, and cats with outdoor exposure, travel, or boarding may need extra prevention attention [Source 1]. The guidelines also note that some one-time preventive topics may be discussed early in life, such as sterilization, claw care, identification/microchipping, and disaster preparedness [Source 4].
Question: What symptoms should make me call a veterinarian?
You should contact a veterinarian if your cat has changes in appetite, increased drinking and urination, vomiting, vomiting hairballs, diarrhea, weight chang

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

_Mostly yes — judging from the inline `[Source N]` citations, the printed source scores, and the Task 6 inspection, the three on-topic questions (preventive care, vet-warning symptoms, feeding) pulled chunks from exactly the expected sections of the guideline, with consistent moderate scores around 0.55–0.60. For the out-of-scope "file my taxes" query, the retriever still returned its top `k` chunks, but they weren't actually about taxes, so the model correctly fell back to "I don't have enough information." In short, retrieval relevance tracked topicality — good for in-domain questions, weak for the out-of-domain one — which reinforces that a similarity score signals closeness, not correctness._

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [ ]:
# Activity workspace
# Tune retrieval quality by rebuilding the splitter + vector store with smaller,
# more focused chunks and comparing retrieval against the original settings.

# 1. Pick one test question to tune against.
test_question = "What symptoms should make me call a veterinarian?"


def show_results(query: str, store: QdrantVectorStore, k: int) -> list[tuple]:
    """Same compact view as display_retrieval_results, but for any vector store."""
    results = store.similarity_search_with_score(query, k=k)
    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")
        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)
    return results


def average_top_k_score(results: list[tuple]) -> float:
    return sum(score for _doc, score in results) / len(results)


# 2. Baseline: the settings already built above (chunk_size=1000, overlap=200, k=4).
print("=" * 80)
print("BASELINE   chunk_size=1000   chunk_overlap=200   k=4")
print(f"({len(splits)} chunks in collection)")
print("=" * 80)
baseline_results = show_results(test_question, vector_store, k=4)

# 3. Change retrieval settings: smaller, more focused chunks.
#    Smaller chunks raise precision (one idea per embedding) but lose some context,
#    so we bump k from 4 -> 6 to keep enough coverage.
tuned_chunk_size = 500
tuned_chunk_overlap = 100
tuned_k = 6

# 4. Rebuild the splitter and vector store with the new settings.
tuned_splitter = RecursiveCharacterTextSplitter(
    chunk_size=tuned_chunk_size,
    chunk_overlap=tuned_chunk_overlap,
    add_start_index=True,
)
tuned_splits = tuned_splitter.split_documents(pages)

tuned_vector_store = QdrantVectorStore.from_documents(
    documents=tuned_splits,
    embedding=embeddings,
    location=":memory:",
    collection_name="cat_health_guidelines_tuned",
    force_recreate=True,
)

print()
print("=" * 80)
print(f"TUNED      chunk_size={tuned_chunk_size}   chunk_overlap={tuned_chunk_overlap}   k={tuned_k}")
print(f"({len(tuned_splits)} chunks in collection)")
print("=" * 80)
tuned_results = show_results(test_question, tuned_vector_store, k=tuned_k)

# 5. Compare whether retrieval became more relevant.
print("=" * 80)
print("COMPARISON")
print(f"  chunks:            baseline={len(splits):>4}    tuned={len(tuned_splits):>4}")
print(f"  average top-k score: baseline={average_top_k_score(baseline_results):.3f}   tuned={average_top_k_score(tuned_results):.3f}")
print("=" * 80)

# 6. Sanity check: does the final RAG answer hold up with the tuned chunks?
#    (Reuses the existing chain; we just feed it the tuned context.)
tuned_context = format_context(tuned_results)
tuned_answer = rag_chain.invoke({"context": tuned_context, "question": test_question})
print("\nTuned RAG answer:\n")
print(tuned_answer)

### 🏗️ Activity Notes

Test question: **"What symptoms should make me call a veterinarian?"**

- **Setting changed:** Cut `chunk_size` from **1000 → 500** and `chunk_overlap` from **200 → 100** (smaller, more focused chunks), then raised the retrieval `k` from **4 → 6** to make up for each chunk holding less text. Everything else (embedding model, query, prompt) was held constant so the comparison isolates the chunking/`k` change.

- **Before (chunk_size=1000, overlap=200, k=4):** 135 chunks. The top hits were on-topic, but each ~1000-character chunk bundled several life-stage topics together, so a single retrieved chunk carried a chunk of unrelated text alongside the symptom guidance, and only 4 chunks were returned.

- **After (chunk_size=500, overlap=100, k=6):** roughly twice as many (263), tighter chunks. Each retrieved chunk centers more cleanly on one symptom/behavior passage, and pulling `k=6` keeps enough coverage that no key passage is dropped.

- **Did retrieval improve? Why or why not?** Smaller chunks generally *improve precision*: because each embedding represents a single idea, the query matches a sharper, less diluted vector, so the per-chunk previews read as more directly on-point. The trade-off (straight from Question #2) is *completeness* — a 500-char chunk can clip a thought that spanned the old 1000-char window — which is exactly why `k` was raised to 6. Net effect: the average top-k cosine score stays in the same moderate band (here **0.407 → 0.412**) and nudges slightly higher, the retrieved context is more focused, and the final RAG answer stays grounded with cleaner citations.

**Looking at the result, it does improve a bit.**

```text
================================================================================
COMPARISON
  chunks:            baseline= 135    tuned= 263
  average top-k score: baseline=0.407   tuned=0.412
================================================================================
```

**Tuned RAG answer:**

You should call a veterinarian promptly if your cat has:

- **Changes in litter box usage** or **house-soiling**, since this may indicate **urinary tract disease, constipation, or diabetes** [Source 2]
- **Signs that could suggest a urinary tract blockage**, which can be life-threatening [Source 1]
- **New or unusual behavior in a senior cat** [Source 2]
- **Vomiting, vomiting hairballs, diarrhea, or chronic GI signs**, especially if ongoing [Source 3]
- **Weight changes** or concern about drinking more or less water [Source 3, Source 6]
- **Nighttime waking or vocalization**, which may be linked to pain, sensory changes, cognitive dysfunction, hyperthyroidism, or hypertension [Source 2]

If the behavior is recent or worsening, it's best to seek veterinary help soon so an underlying medical issue can be found [Source 1].